# Taking Input
Giving the user two choices of input.
They can either paste the URL of a website containing the article like Wikipedia or they can input the text to be summarised through the input field.

In [ ]:
print("Please choose your prefered way to input text.\nPress 1 to copy paste the URL of a website.\nPress 2 to input your own text.\n")
input_choice = input()
summary_size = (input("Enter the lines you want in the summary.\n"))
if input_choice == "1":
    url = input("\nPlease enter the URL:\n")
    #importing the necessary libraries to parse the data from the url
    import bs4 as bs
    import urllib.request
    import re
    headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/15.0 Safari/605.1.15'}
    req = urllib.request.Request(url, headers=headers)
    scraped_data = urllib.request.urlopen(req) 
    #reading the data from the object returned by the urlopen
    article = scraped_data.read()
    #parsing the article using BeautifulSoup
    parsed_article = bs.BeautifulSoup(article,'lxml')
    #using findall function on the object returned by BeautifulSoup to get the paragraphs
    paragraphs = parsed_article.find_all('p')
    article_text = ""
    for p in paragraphs:
        article_text += p.text
else:
 article_text = input("Please enter your desired text.\n")

Please choose your prefered way to input text.
Press 1 to copy paste the URL of a website.
Press 2 to input your own text.



HTTPError: HTTP Error 403: Forbidden

Ensure article_text is not empty


In [23]:
if not article_text.strip():
    raise ValueError("The input text is empty. Please provide valid text.")

In [27]:
#importing PythonRegex(Regular Expression)
import re
# Removing Square Brackets and Extra Spaces and replacing them with white spaces
article_text = re.sub(r'\[[0-9]*\]', ' ', article_text)  
article_text = re.sub(r'\s+', ' ', article_text) 
article_text = re.sub(r'[^\x00-\x7F]+', ' ', article_text)  # Remove non-ASCII characters
article_text = re.sub(r'\s+', ' ', article_text)  # Remove extra spaces

In [31]:
# Removing special characters and digits
formatted_article_text = re.sub('[^a-zA-Z]', ' ', article_text )  
formatted_article_text = re.sub(r'\s+', ' ', formatted_article_text) 
print("Formatted Article Text:", formatted_article_text)


Formatted Article Text: Help Advanced SearcharXivLabs is a framework that allows collaborators to develop and share new arXiv features directly on our website Both individuals and organizations that work with arXivLabs have embraced and accepted our values of openness community excellence and user data privacy arXiv is committed to these values and only works with partners that adhere to them Have an idea for a project that will add value for arXiv s community Learn more about arXivLabs arXiv Operational Status Get status notifications via email or slack 


# Conveting the processed text into sentences 

In [33]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
try:
    sentence_list = nltk.sent_tokenize(article_text)
    print("Sentence List:", sentence_list)
except Exception as e:
    print("Error in nltk.sent_tokenize:", str(e))
#sentence_list = nltk.sent_tokenize(article_text) 

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/kashyaphebbar/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/kashyaphebbar/nltk_data...


Sentence List: ['Help | Advanced SearcharXivLabs is a framework that allows collaborators to develop and share new arXiv features directly on our website.Both individuals and organizations that work with arXivLabs have embraced and accepted our values of openness, community, excellence, and user data privacy.', "arXiv is committed to these values and only works with partners that adhere to them.Have an idea for a project that will add value for arXiv's community?", 'Learn more about arXivLabs.', 'arXiv Operational Status Get status notifications via email or slack']


[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


# Finding the frequency of occurency of words along with their weights

In [34]:
import nltk
nltk.download('stopwords')


stopwords = nltk.corpus.stopwords.words('english')

word_frequencies = {}  
for word in nltk.word_tokenize(formatted_article_text):  
    if word not in stopwords:
        if word not in word_frequencies.keys():
            word_frequencies[word] = 1
        else:
            word_frequencies[word] += 1

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/kashyaphebbar/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [35]:
maximum_frequncy = max(word_frequencies.values())

for word in word_frequencies.keys():  
    word_frequencies[word] = (word_frequencies[word]/maximum_frequncy)

# Calculating the scores of the sentences

In [36]:
sentence_scores = {}  
for sent in sentence_list:  
    for word in nltk.word_tokenize(sent.lower()):
        if word in word_frequencies.keys():
            if len(sent.split(' ')) < 30:
                if sent not in sentence_scores.keys():
                    sentence_scores[sent] = word_frequencies[word]
                else:
                    sentence_scores[sent] += word_frequencies[word]

# Getting the Summary

In [39]:
import heapq

# Ensure summary_size is an integer
summary_size = int(summary_size)

# Debug: Check sentence_scores
print("Sentence Scores:", sentence_scores)

# Ensure sentence_scores is not empty
if not sentence_scores:
    raise ValueError("No sentence scores were calculated. Ensure the input text is valid.")

# Limit summary_size to the number of sentences
summary_size = min(summary_size, len(sentence_scores))

# Get the summary
summary_sentences = heapq.nlargest(summary_size, sentence_scores, key=sentence_scores.get)
summary = ' '.join(summary_sentences)
print("Summary:", summary)

Sentence Scores: {"arXiv is committed to these values and only works with partners that adhere to them.Have an idea for a project that will add value for arXiv's community?": 3.0, 'arXiv Operational Status Get status notifications via email or slack': 1.5}
Summary: arXiv is committed to these values and only works with partners that adhere to them.Have an idea for a project that will add value for arXiv's community? arXiv Operational Status Get status notifications via email or slack
